# Aula 10: pipeline de dados com PySpark

Atividade prática de um pipeline de engenharia de dados completo, usando o
Spark com a base de fraude em cartão (a mesma das Aulas 2, 5, 6, 7, 8 e 9),
em escala de 1 milhão de transações. O pipeline tem seis passos: extract,
transform com quarentena, unify, bifurcação por regra de negócio, load em
múltiplos destinos, e reconciliação final.

Rode as células em ordem, uma por vez. Onde aparecer `# TODO`, complete a
linha antes de rodar. Se travar, escreva no chat ou chame os professores com
o número do passo em que travou.

## Passo 0 · Preparar o ambiente

O Google Colab não vem com Spark instalado. A célula abaixo resolve isso, e
roda só uma vez por sessão (se o Colab desconectar, rode de novo desde o
início).

In [ ]:
!apt-get install -y openjdk-17-jdk-headless -qq > /dev/null
!pip install pyspark -q

import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("aula10").getOrCreate()
spark

In [ ]:
import gdown
import os

# Dicionário com IDs dos arquivos no Google Drive
files = {
    'cartoes.csv': '13Xf9ILhqxaIcvjq0vENJFCPD-HX-FLjM',
    'categorias.csv': '1NeAB_U66QnzRHZ1tJesP5lZ0jccXLAke',
    'clientes.csv': '1zgr0k17FdKp5aRkSd96qtLyJ9x0b4bFv',
    'contestacoes.csv': '1lSpAe8stD3JEMDJMFbiiYrQH0jYtHzT8',
    'lojistas.csv': '1vydqkbd2fONvNAfNKon9izr0pqaGBqhy',
    'transacoes.csv': '1OR3lgLjqVQZoajau2wrSExWPYc8xUsFS'
}

# Criar pasta para os CSVs para manter a estrutura
os.makedirs('csv', exist_ok=True)

print("Baixando arquivos...")
for filename, file_id in files.items():
    url = f'https://drive.google.com/uc?id={file_id}'
    output = f'csv/{filename}' if filename.endswith('.csv') else filename
    gdown.download(url, output, quiet=True)
    print(f"- {filename} baixado.")

PASTA_DADOS = "/content/csv"

## Passo 1 · Extract

As seis tabelas do caso de fraude, direto do CSV.

In [ ]:
from pyspark.sql import functions as F

dict_dfs = {}

categorias   = spark.read.csv(f"{PASTA_DADOS}/categorias.csv",   header=True, inferSchema=True)
dict_dfs['categorias'] = categorias

clientes     = spark.read.csv(f"{PASTA_DADOS}/clientes.csv",     header=True, inferSchema=True)
dict_dfs['clientes'] = clientes

lojistas     = spark.read.csv(f"{PASTA_DADOS}/lojistas.csv",     header=True, inferSchema=True)
dict_dfs['lojistas'] = lojistas

cartoes      = spark.read.csv(f"{PASTA_DADOS}/cartoes.csv",      header=True, inferSchema=True)
dict_dfs['cartoes'] = cartoes

transacoes   = spark.read.csv(f"{PASTA_DADOS}/transacoes.csv",   header=True, inferSchema=True)
dict_dfs['transacoes'] = transacoes

contestacoes = spark.read.csv(f"{PASTA_DADOS}/contestacoes.csv", header=True, inferSchema=True)
dict_dfs['contestacoes'] = contestacoes

for name, df in dict_dfs.items():
    print(f"Schema do DataFrame {name}")
    df.printSchema()
    print()


## Passo 2 · Transform, com quarentena

Nas versões anteriores deste notebook, linhas problemáticas eram descartadas
direto com `.dropna()` e `.filter()`. Isso funciona, mas apaga a evidência: se
alguém perguntar depois "por que sumiram 40 mil linhas?", não tem como
responder.

A versão de hoje marca cada transação com o motivo do problema, em vez de
simplesmente descartar, e separa duas tabelas: `transacoes_validas` e
`transacoes_rejeitadas`. Isso é o padrão de **quarentena de dados**: nada se
perde sem deixar rastro.

In [ ]:
# Conta quantas vezes cada id_transacao aparece, para detectar duplicidade
contagem_por_id = transacoes.groupBy("id_transacao").agg(F.count("*").alias("qtd_ocorrencias"))
transacoes_marcada = transacoes.join(contagem_por_id, on="id_transacao", how="left")

transacoes_marcada = transacoes_marcada.withColumn(
    "motivo_rejeicao",
    F.when(
        F.col("id_transacao").isNull() | F.col("id_cartao").isNull() |
        F.col("id_lojista").isNull() | F.col("valor").isNull(),
        F.lit("campo obrigatorio nulo")
    )
     .when(F.col("qtd_ocorrencias") > 1, F.lit("id_transacao duplicado"))
     .when(F.col("valor") <= 0, F.lit("valor nao positivo"))
     .otherwise(F.lit(None))
)

transacoes_validas = (
    transacoes_marcada
    .filter(F.col("motivo_rejeicao").isNull())
    .drop("qtd_ocorrencias", "motivo_rejeicao")
)
transacoes_rejeitadas = transacoes_marcada.filter(F.col("motivo_rejeicao").isNotNull())

print("Linhas extraidas:", transacoes.count())
print("Linhas validas:", transacoes_validas.count())
print("Linhas em quarentena:", transacoes_rejeitadas.count())

transacoes_rejeitadas.coalesce(1).write.mode("overwrite").option("header", True).csv("/content/saida/rejeitadas")

Se `transacoes_rejeitadas` deu zero linhas, não é bug: a base desta turma já
chega limpa nesses três critérios. Isso não torna a quarentena inútil. Ela
existe para o dia em que uma carga futura chegar suja, e nesse dia ninguém vai
precisar reescrever o pipeline, só olhar a pasta `rejeitadas` para entender o
que aconteceu.

## Passo 3 · Unify, primeira tentativa

Junte `transacoes_validas` com as outras quatro tabelas, pela chave que liga
cada par. Depois, rode a célula seguinte para conferir o nome do cliente numa
das transações.

In [ ]:
completo = (
    transacoes_validas
    .join(cartoes, on="id_cartao", how="inner")
    .join(clientes, on="id_cliente", how="inner")
    .join(lojistas, on="id_lojista", how="inner")
    .join(categorias, on="id_categoria", how="left")
)

In [ ]:
completo.select("nome").show(5)

Se a célula acima deu um erro mencionando `nome` como referência ambígua
(*ambiguous reference*), não é bug seu. `clientes`, `lojistas` e
`categorias` têm, cada uma, uma coluna chamada `nome`, com um significado
diferente em cada tabela. O Spark não sabe qual das três você quis dizer.

Isso é exatamente o problema de **consistência** que a Aula 9 apresentou:
o mesmo nome de campo significando coisas diferentes em tabelas diferentes.

A saída é renomear a coluna `nome` de cada tabela antes de unir, com
`.alias(...)`, para que o resultado final tenha `nome_cliente`,
`nome_lojista` e `nome_categoria`, cada um com um nome só seu.

In [ ]:
clientes_r = clientes.select(
    "id_cliente",
    F.col("nome").alias("nome_cliente"),
)

lojistas_r = lojistas.select(
    "id_lojista",
    "id_categoria",
    F.col("nome").alias("nome_lojista"),
)

categorias_r = categorias.select(
    "id_categoria",
    F.col("nome").alias("nome_categoria"),
)

## Passo 3 (continuação) · Unify, com os nomes corrigidos

Refaça a união, agora usando `clientes_r`, `lojistas_r` e `categorias_r` no
lugar das tabelas originais. De `cartoes`, use só as três colunas
necessárias, para não repetir nenhuma outra coluna por acidente. Guarde
também `status`, que vai definir o próximo passo.

In [ ]:
base_unificada = (
    transacoes_validas
    .join(cartoes.select("id_cartao", "id_cliente", "bandeira"), on="id_cartao", how="inner")
    .join(clientes_r, on="id_cliente", how="inner")
    .join(lojistas_r, on="id_lojista", how="inner")
    .join(categorias_r, on="id_categoria", how="left")
)

base_unificada.select("id_transacao", "valor", "status", "nome_cliente", "nome_lojista", "nome_categoria").show(5)

## Passo 4 · Bifurcação por regra de negócio

Nem toda transação segue o mesmo caminho depois de unificada. A coluna
`status` já separa três situações reais: `aprovada`, `recusada` e
`contestada`. Faz sentido tratar isso como duas trilhas diferentes, cada
uma indo para um destino diferente:

- **Trilha normal**: transações aprovadas, seguem para o resumo de vendas
  por lojista.
- **Trilha de atenção**: transações recusadas ou contestadas, seguem para um
  time de risco revisar, enriquecidas com o detalhe da contestação quando
  existir.

In [ ]:
trilha_normal = base_unificada.filter(F.col("status") == "aprovada")

trilha_atencao = base_unificada.filter(F.col("status").isin("recusada", "contestada"))

trilha_atencao = trilha_atencao.join(
    contestacoes.select("id_transacao", "motivo", "resultado"),
    on="id_transacao",
    how="left",
)

print("Trilha normal:", trilha_normal.count())
print("Trilha de atencao:", trilha_atencao.count())
print("Soma das duas trilhas:", trilha_normal.count() + trilha_atencao.count())
print("Base unificada:", base_unificada.count())

Repare que parte da `trilha_atencao` (as `recusada`) vai ter `motivo` e
`resultado` nulos depois do join. Não é erro: nem toda transação recusada
teve uma contestação formal aberta por um cliente. É o `LEFT JOIN`
funcionando exatamente como deveria, trazendo o detalhe só de quem tem.

## Passo 5 · Load, em múltiplos destinos

Três saídas diferentes, para três consumidores diferentes:

1. A trilha normal completa, particionada em Parquet, como um data lake de
   verdade organizaria.
2. A trilha de atenção, num CSV único, pequeno o bastante para o time de
   risco abrir direto no Sheets.
3. Um resumo agregado por lojista, também em CSV único, para quem só quer
   o número final.

In [ ]:
(
    trilha_normal
    .write.mode("overwrite")
    .partitionBy("estado")
    .parquet("/content/saida/normal")
)

(
    trilha_atencao
    .coalesce(1)
    .write.mode("overwrite").option("header", True)
    .csv("/content/saida/atencao")
)

(
    trilha_normal
    .groupBy("nome_lojista")
    .sum("valor")
    .orderBy(F.desc("sum(valor)"))
    .coalesce(1)
    .write.mode("overwrite").option("header", True)
    .csv("/content/saida/resumo_por_lojista")
)

## Passo 6 · Reconciliação

Antes de considerar o pipeline pronto, prove que nenhuma linha sumiu sem
explicação em nenhuma etapa. Isso é rotina em engenharia de dados de
verdade: contagem bate, ou alguém precisa explicar por quê.

In [ ]:
print("Extraidas:", transacoes.count())
print("Validas + rejeitadas:", transacoes_validas.count() + transacoes_rejeitadas.count())
print("Base unificada:", base_unificada.count())
print("Trilha normal + trilha de atencao:", trilha_normal.count() + trilha_atencao.count())

assert transacoes.count() == transacoes_validas.count() + transacoes_rejeitadas.count(), \
    "perdeu ou duplicou linha na quarentena"
assert base_unificada.count() == trilha_normal.count() + trilha_atencao.count(), \
    "perdeu ou duplicou linha na bifurcacao"

print("Reconciliacao OK: nenhuma linha sumiu sem explicacao.")

## Perguntas de conferência
### Pergunta 1: Quantas transações ficaram em quarentena? E quantas foram para a trilha de atenção?

In [ ]:
print("Em quarentena:", transacoes_rejeitadas.count())
print("Na trilha de atencao:", trilha_atencao.count())

### Pergunta 2: Qual lojista teve o maior valor total transacionado, considerando só a trilha normal?

In [ ]:
(
    trilha_normal
    .groupBy("nome_lojista")
    .sum("valor")
    .orderBy(F.desc("sum(valor)"))
    .show(5)
)

## Pronto

Confira se os seis passos rodaram sem erro (extract, transform com
quarentena, unify, bifurcação, load em múltiplos destinos e reconciliação),
e anote as respostas das perguntas de conferência. Seu instrutor vai avisar
como entregar. Se alguma célula travou, escreva no chat com o número do
passo.